# Notebook 02 — Classical Baseline Models

**PharmaSentinel** | Harshita Adlakha

Establishes strong classical baselines using TF-IDF + sklearn pipelines.
These populate Table 2 of the paper and serve as the lower bound for
the BERT benchmark in Notebook 03.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

from pharmasentinel.data import load_raw_data, preprocess, build_splits
from pharmasentinel.models import (
    logistic_regression_baseline, svm_baseline, random_forest_baseline,
    ridge_regression_baseline, run_baselines, BASELINE_REGISTRY,
)
from pharmasentinel.training import (
    sentiment_metrics, condition_metrics, rating_metrics, build_benchmark_table
)
from pharmasentinel.utils import set_seed

set_seed(42)
sns.set_theme(style='whitegrid')
print('Ready.')

In [ ]:
raw = load_raw_data('../data/drugsComTrain_raw.tsv', '../data/drugsComTest_raw.tsv')
df, cond_enc, _ = preprocess(raw, top_k_conditions=50)
train_df, val_df, test_df = build_splits(df)

X_train = train_df['review_clean'].tolist()
X_test  = test_df['review_clean'].tolist()

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

## Sentiment Classification Baselines

In [ ]:
sent_results = run_baselines(
    X_train, train_df['sentiment'].values,
    X_test,  test_df['sentiment'].values,
    task='sentiment',
)
print(build_benchmark_table(sent_results))

In [ ]:
# Confusion matrix for best baseline (SVM)
best_model = svm_baseline()
best_model.fit(X_train, train_df['sentiment'].values)
y_pred = best_model.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    test_df['sentiment'].values, y_pred,
    display_labels=['Negative', 'Neutral', 'Positive'],
    cmap='Blues', ax=ax
)
ax.set_title('SVM Confusion Matrix (Sentiment)', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/baseline_sentiment_cm.png', bbox_inches='tight')
plt.show()

print(classification_report(test_df['sentiment'].values, y_pred,
                            target_names=['Negative', 'Neutral', 'Positive']))

## Condition Classification Baselines

In [ ]:
cond_results = run_baselines(
    X_train, train_df['condition_label'].values,
    X_test,  test_df['condition_label'].values,
    task='condition',
)
print(build_benchmark_table(cond_results))

## Rating Regression Baselines

In [ ]:
ridge = ridge_regression_baseline()
ridge.fit(X_train, train_df['rating_norm'].values)
y_pred_r = ridge.predict(X_test)

from pharmasentinel.training import rating_metrics
print(rating_metrics(test_df['rating_norm'].values, y_pred_r))

# Predicted vs actual scatter
fig, ax = plt.subplots(figsize=(6, 5))
sample_idx = np.random.choice(len(y_pred_r), size=3000, replace=False)
ax.scatter(
    test_df['rating_norm'].values[sample_idx] * 9 + 1,
    y_pred_r[sample_idx] * 9 + 1,
    alpha=0.2, s=10, color='steelblue'
)
ax.plot([1, 10], [1, 10], 'r--', lw=1.5, label='Perfect')
ax.set_xlabel('Actual Rating')
ax.set_ylabel('Predicted Rating')
ax.set_title('Ridge Regression: Actual vs Predicted', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/baseline_rating_scatter.png', bbox_inches='tight')
plt.show()

## Summary

Classical baselines achieve decent sentiment F1 (~0.80) but struggle with condition classification (~0.71 accuracy across 51 classes). This performance gap motivates the move to pre-trained language models in Notebook 03.